# 12 — Exception Handling

## Objectives
- Understand Java's exception hierarchy
- Use try-catch-finally, multi-catch, try-with-resources
- Create custom exceptions for domain-specific errors
- Apply exception chaining

## Exception Hierarchy
```
Throwable
├── Error (JVM errors — don't catch)
│   ├── OutOfMemoryError
│   └── StackOverflowError
└── Exception
    ├── RuntimeException (Unchecked)
    │   ├── NullPointerException
    │   ├── IllegalArgumentException
    │   └── ArrayIndexOutOfBoundsException
    └── IOException (Checked — must handle)
        └── FileNotFoundException
```

In [1]:
// Custom exception hierarchy for banking
class BankException extends Exception {
    private final String errorCode;
    BankException(String code, String msg) { super(msg); this.errorCode = code; }
    BankException(String code, String msg, Throwable cause) { super(msg, cause); this.errorCode = code; }
    String getErrorCode() { return errorCode; }
}

class InsufficientFundsException extends BankException {
    private final double shortfall;
    InsufficientFundsException(double requested, double available) {
        super("INSUF-FUNDS", String.format("Need INR %.2f more. Available: INR %.2f", 
              requested - available, available));
        this.shortfall = requested - available;
    }
    double getShortfall() { return shortfall; }
}

class AccountFrozenException extends BankException {
    AccountFrozenException(String accountId) {
        super("ACC-FROZEN", "Account " + accountId + " is frozen. Contact support.");
    }
}

// Banking service with exception handling
class BankingService {
    private double balance = 10000;
    private boolean frozen = false;
    private String accountId = "ACC-001";
    
    void withdraw(double amount) throws BankException {
        if (frozen) throw new AccountFrozenException(accountId);
        if (amount > balance) throw new InsufficientFundsException(amount, balance);
        balance -= amount;
        System.out.printf("Withdrawn INR %.2f. Balance: INR %.2f%n", amount, balance);
    }
    
    void freeze() { frozen = true; }
}

BankingService svc = new BankingService();

// Test normal withdrawal
try { svc.withdraw(2000); } catch (BankException e) { System.out.println("Error: " + e.getMessage()); }

// Test insufficient funds
try { svc.withdraw(20000); }
catch (InsufficientFundsException e) {
    System.out.println("Insufficient! Short by: INR " + e.getShortfall());
    System.out.println("Error code: " + e.getErrorCode());
}
catch (BankException e) { System.out.println("Bank error: " + e.getMessage()); }

// Test frozen account
svc.freeze();
try { svc.withdraw(100); }
catch (AccountFrozenException e) { System.out.println("FROZEN: " + e.getMessage()); }
catch (BankException e) { System.out.println("Error: " + e.getMessage()); }

Withdrawn INR 2000.00. Balance: INR 8000.00
Insufficient! Short by: INR 12000.0
Error code: INSUF-FUNDS
FROZEN: Account ACC-001 is frozen. Contact support.


## Mini Challenge
Create `InvalidProductException`, `OutOfStockException`, `PaymentDeclinedException` for an e-commerce system.

In [4]:
// 1. InvalidProductException - Thrown when a product ID or data is malformed/invalid
public class InvalidProductException extends Exception {
    public InvalidProductException(String message) {
        super(message);
    }
}

// 2. OutOfStockException - Thrown when a user tries to buy an item that is sold out
public class OutOfStockException extends Exception {
    public OutOfStockException(String message) {
        super(message);
    }
}

// 3. PaymentDeclinedException - Thrown when the payment gateway rejects the transaction
public class PaymentDeclinedException extends Exception {
    public PaymentDeclinedException(String message) {
        super(message);
    }
}

In [5]:
public class ECommerceSystem {
    
    // Mock checkout method that demonstrates throwing the custom exceptions
    public static void processOrder(String productId, int quantity, double balance) 
            throws InvalidProductException, OutOfStockException, PaymentDeclinedException {
        
        if (productId == null || productId.isEmpty()) {
            throw new InvalidProductException("Product ID cannot be null or empty.");
        }
        
        if (quantity > 10) { // Assuming 10 is max stock for this test
            throw new OutOfStockException("Requested quantity exceeds available stock.");
        }
        
        
        if (balance < 50.0) { // Assuming order total is $50
            throw new PaymentDeclinedException("Insufficient funds. Transaction declined.");
        }
        
        System.out.println("Order processed successfully!");
    }

    public static void main(String[] args) {
        // Test Case: Out of Stock
        try {
            processOrder("PROD123", 15, 100.0);
        } catch (InvalidProductException | OutOfStockException | PaymentDeclinedException e) {
            System.err.println("Checkout Error: " + e.getMessage());
        }
    }
}

ECommerceSystem.main(new String[]{});

Checkout Error: Requested quantity exceeds available stock.
